In [ ]:
# !pip install sentence-transformers
# !pip install chromadb
# !pip install pdfplumber
# !pip install langchain_text_splitters
# !pip install spacy
# !python -m spacy download pt_core_news_sm

In [ ]:
import chromadb
import spacy
import re
from sentence_transformers import SentenceTransformer

In [ ]:
def carregar_pdf(caminho_arquivo):
    """Carrega e extrai texto de um arquivo PDF"""
    import pdfplumber
    texto_completo = []
    with pdfplumber.open(caminho_arquivo) as pdf:
        for pagina in pdf.pages:
            texto = pagina.extract_text()
            if texto:
                texto_completo.append(texto)
    return "\n".join(texto_completo)

In [ ]:
nlp = spacy.load('pt_core_news_sm')

def limpar_texto(texto):
    """Realiza limpeza básica do texto"""
    texto = texto.lower()
    texto = re.sub(r'\n\n', ' \n\n', texto) # manter quebras de parágrafo em caso de recursive chunking
    texto = re.sub(r'\n', ' \n', texto)
    texto = re.sub(r'\.', ' .', texto)
    texto = re.sub(r'[^a-z0-9áàâãéêíóôõúüçñ\s\n.]', ' ', texto) # ^ indica negação
    texto = re.sub(r'  +', ' ', texto) # multiplos espaços
    return texto.strip()

def tokenizar_spacy(texto):
    """Tokeniza o texto usando spaCy"""
    doc = nlp(texto)
    return [token for token in doc]

def processar_texto(texto):
    """Pipeline completo de PLN"""
    clean_text = limpar_texto(texto)
    tokens = tokenizar_spacy(clean_text)
    return ' '.join([token.text for token in tokens])

In [ ]:
texto = carregar_pdf("../../docs/datasets/2022 AHAACCHFSA.pdf")
#texto = carregar_pdf("../../docs/datasets/Brazilian Guidelines of Hypertension – 2020.pdf")
#texto = carregar_pdf("../../docs/datasets/PROTOCOLOS DE USO DO GUIA ALIMENTAR PARA A POPULAÇÃO BRASILEIRA.pdf")
#texto = carregar_pdf("../../docs/datasets/Terapia Nutricional no Diabetes tipo 1.pdf")
texto_processado = processar_texto(texto)
print(texto_processado)

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=150,
    chunk_overlap=20,
    separators=["\n\n", "\n", ".", " ", ""] # '.' tem mais valor semântico que '\n' pelo modo como o arquivo(pdf) é carregado
)

chunks = recursive_splitter.split_text(texto_processado)

for index, chunk in enumerate(chunks):
    chunks[index] = re.sub(r' \n', '', chunks[index])
    chunks[index] = re.sub(r'\. ', '', chunks[index])
    chunks[index] = re.sub(r' \.', '', chunks[index])
    #chunk = re.sub(r'\,', '', chunk)

print("n_chunks: ", len(chunks))
print(chunks)

In [ ]:
# ENCODE_MODEL = "paraphrase-multilingual-MiniLM-L12-v2"
ENCODE_MODEL = 'all-MiniLM-L6-v2' # leve

modelo = SentenceTransformer(ENCODE_MODEL)

def criar_embeddings(chunks):
    """Cria embeddings para os chunks"""
    embeddings = modelo.encode(chunks, show_progress_bar=True)
    return embeddings

In [ ]:
client = chromadb.PersistentClient(path='./chroma_db')
    
collection = client.get_or_create_collection(name='nutrition', embedding_function=None)

print(f"ChromaDB inicializado: {collection.count()} documentos na coleção")

In [ ]:
if collection.count() > 0:
    client.delete_collection(name='nutrition')

In [ ]:
# Criar ids únicos
ids = [f"chunk_{i}" for i in range(len(chunks))]

embeddings = criar_embeddings(chunks)

collection.add(
    ids=ids,
    documents=chunks,
    embeddings=embeddings.tolist()  # converter para lista
)

In [ ]:
questions = ["?"]

for question in questions:
    question_embedding = modelo.encode([question])
    resultados = collection.query(
        query_embeddings=question_embedding.tolist(),
        n_results=3
    )
    print(f"\nPergunta: {question}")
    for doc, score in zip(resultados['documents'][0], resultados['distances'][0]):
        print(f" - Documento: {doc} (Score: {score})")